# ♻️ PreZero / Fillinq: Glass Collection Analysis
### Zakaria’s exploratory workbook — corrected observation semantics

The original material-comparison, trajectory, calendar and grouping structure is retained. Values describe recorded pre-emptying visits. The four numbered notebooks extend this analysis through demand projection and Gurobi routing. Old overflow=110%, drop-detected emptying and asserted savings interpretations have been withdrawn.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "Zakaria-Work"
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from Zakaria-Work or its parent directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loader import load_and_clean_data, collection_summary
from src.clustering import extract_container_profiles, cluster_containers
from src.predictor import predict_container_demands, select_collection_sites
OUT = ROOT / "data" / "corrected"
OUT.mkdir(parents=True, exist_ok=True)
MAT_COLORS = {"white": "#1f77b4", "green": "#2ca02c", "brown": "#8c564b"}
plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 100})


## 2. Data ingestion and location identity
UUIDs identify anonymized locations; each location/material pair is a separate bin. Real reference-map points are not a recovered mapping to these UUIDs. Numeric fill and explicit overflow remain distinct.

In [ ]:
df = load_and_clean_data(ROOT / "glass_fill_levels_anonymized_f.csv")
print(f"{len(df):,} pre-emptying observations; {df.Location_id.nunique()} locations; "
      f"{len(df.groupby(['Location_id', 'material_type']))} observed bins")
print(f"Coverage: {df.date.min().date()} to {df.date.max().date()}")
display(df.head())


In [ ]:
from src.distance_provider import ModularDistanceProvider
location_ids = sorted(df.Location_id.unique())
provider = ModularDistanceProvider(location_ids)
locations_df = pd.DataFrame({'Location_id':location_ids, 'lat':provider.coords[:,0], 'lon':provider.coords[:,1], 'is_synthetic':True})
print('Illustrative coordinate assignment only:', provider.distance_provenance)
display(locations_df.head())


In [ ]:
display(collection_summary(df))
print('Numeric means omit explicit overfull; every row is a pre-emptying visit.')


---
## 📊 3. Exploratory Data Analysis & Overall Distributions

Let's look at the overall distribution of fill levels and the proportion of readings across the three glass categories: **Brown**, **Green**, and **White**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Fill Level Distribution
fl_counts = df['fill_level_perc'].value_counts()
ordered_cats = ['0', '10', '20', '30', '40', '50', '60', '70', '80', '90', '100', 'overfull']
fl_ordered = fl_counts.reindex(ordered_cats)

bar_colors = ['#2ca02c' if c in ['10', '20', '30', '40'] else '#ff7f0e' if c in ['50', '60', '70', '80'] else '#d62728' for c in ordered_cats]

sns.barplot(x=fl_ordered.index, y=fl_ordered.values, ax=axes[0], palette=bar_colors)
axes[0].set_title("Overall Fill Level Distribution (Frequency)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Recorded Fill Level (%)", fontsize=11)
axes[0].set_ylabel("Count of Observations", fontsize=11)
for i, v in enumerate(fl_ordered.values):
    axes[0].text(i, v + 25, f"{v:,}", ha='center', fontsize=9)

# 2. Material Breakdown
mat_counts = df['material_type'].value_counts()
sns.barplot(x=mat_counts.index, y=mat_counts.values, ax=axes[1], palette=[MAT_COLORS[m] for m in mat_counts.index])
axes[1].set_title("Total Observations by Material Type", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Glass Material Type", fontsize=11)
axes[1].set_ylabel("Count of Observations", fontsize=11)
for i, (m, v) in enumerate(mat_counts.items()):
    axes[1].text(i, v + 40, f"{v:,} ({v/len(df)*100:.1f}%)", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Material trajectories across the observed period
Daily means and IQRs summarize bins visited that day. Individual points are pre-emptying observations separated by resets. Connecting them helps inspect history; it does not reconstruct a physical sawtooth or daily network occupancy.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
for ax, mat in zip(axes, ['white','green','brown']):
    sub = df[df.material_type.eq(mat)]
    daily = sub.groupby('date').fill_num.agg(mean='mean', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75))
    ax.fill_between(daily.index, daily.q25.to_numpy(dtype=float), daily.q75.to_numpy(dtype=float), color=MAT_COLORS[mat], alpha=.2, label='Numeric visit IQR')
    ax.plot(daily.index,daily['mean'],color=MAT_COLORS[mat],label='Numeric visit mean')
    chosen = sub.groupby('Location_id').size().nlargest(1).index
    for loc in chosen:
        history = sub[sub.Location_id.eq(loc)]
        ax.scatter(history.date,history.fill_num,s=16,label=f'Observed bin {loc[:8]}')
    ax.set(title=f'{mat.capitalize()} glass: recorded visits', ylabel='Numeric fill (%)', ylim=(-2,102))
    ax.legend()
plt.tight_layout()
plt.show()


### Reading these plots
Compare material patterns, sparse coverage and visit timing. Full is numeric 100%; explicit overfull is reported separately. Neither monthly means nor connected visit points prove filling speed, holiday causality or avoidable collections.

In [ ]:
stream_comparison_df = collection_summary(df)
stream_comparison_df['overfull_share_pct'] = (100 * stream_comparison_df.overfull_count / stream_comparison_df.collections)
display(stream_comparison_df.round(2))
print("Means exclude overfull. Overfull shares use every collection observation as denominator.")


## 5. Calendar dynamics
Public holidays and the festive window are external descriptive labels. Compare recorded activity and numeric means without assigning an unmeasured seasonal cause.

In [ ]:
dow = df.groupby('day_name').agg(readings=('date','size'), numeric_mean=('fill_num','mean')).reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
monthly = df.pivot_table(index='year_month', columns='material_type', values='fill_num', aggfunc='mean')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
dow.readings.plot.bar(ax=axes[0], color='#4676a3')
axes[0].set(title='Recorded collection activity by weekday', ylabel='Observations')
monthly.plot(ax=axes[1], marker='o', color=[MAT_COLORS[m] for m in monthly.columns])
axes[1].set(title='Monthly numeric pre-emptying fill', ylabel='Numeric mean (%)', xlabel='Month')
plt.tight_layout()
plt.show()
display(df.groupby('is_public_holiday').agg(readings=('date','size'), numeric_mean=('fill_num','mean'), overfull_share=('is_overfull','mean')))
print("Calendar comparisons are descriptive: visit timing and coverage can confound them.")


## 6. Collection-state audit
Count all recorded pre-emptying readings. Do not infer collections from decreases between readings. Low-fill observations can motivate an experiment but do not establish why the operator visited or that the visit was unnecessary.

In [ ]:
audit = pd.DataFrame({
    'Recorded state': ['Numeric 0–30%', 'Numeric above 30/below 70%', 'Numeric 70/below 100%', 'Numeric 100%', 'Explicit overfull'],
    'Collections': [df.fill_num.le(30).sum(), (df.fill_num.gt(30)&df.fill_num.lt(70)).sum(),
                    (df.fill_num.ge(70)&df.fill_num.lt(100)).sum(), df.fill_num.eq(100).sum(), df.is_overfull.sum()]
})
audit['Share of collections (%)'] = audit.Collections / len(df) * 100
display(audit.round(2))
ax = audit.plot.barh(x='Recorded state', y='Collections', legend=False, color='#4676a3')
ax.set_title('What was observed immediately before each recorded emptying')
plt.tight_layout()
plt.show()
print("These bands describe visits; they do not prove a collection was avoidable.")


## 7. Bin profiles and descriptive groups
Profiles preserve location/material identity and use only data strictly before the chosen dispatch date. Completed-cycle rate medians replace positive differences. These remain transparent lower-bound proxies for censored cycles.

In [ ]:
# Change this explicit morning dispatch date to replay a different day.
PLANNING_DATE = pd.Timestamp("2026-08-17")
profiles = extract_container_profiles(df, as_of=PLANNING_DATE, inclusive=False)
clustered_df, kmeans_model = cluster_containers(profiles, n_clusters=4)
print(f"Known bins strictly before dispatch: {len(profiles)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, sub in clustered_df.groupby('tier'):
    axes[0].scatter(sub.avg_fill_perc, sub.overfull_rate, label=f"{name} (n={len(sub)})", alpha=0.65)
axes[0].set(xlabel='Mean numeric pre-emptying fill (%)', ylabel='Explicit overfull observations (%)', title='Behavioral groups at the planning cutoff')
axes[0].legend(fontsize=8)
sns.boxplot(data=clustered_df, x='material_type', y='daily_fill_velocity', ax=axes[1])
axes[1].set(xlabel='Glass material', ylabel='Cycle-median rate proxy (pp/day)', title='Rates from completed filling cycles')
plt.tight_layout()
plt.show()


## 8. Configurable operating assumptions
The modular route notebook enforces volume, payload, shift, per-bin service, final unloading and yard return. Density 350 kg/m³, truck volume 18 m³ and depot/disposal locations remain assumptions. Multi-chamber limits and breaks are not silently represented by unused configuration objects.

In [ ]:
operations = pd.DataFrame({'Variable':['Container volume','Assumed glass bulk density','Assumed truck volume','Truck payload','Shift','Service per bin','Assumed unloading'],
    'Value':[3,350,18,12000,8,5,15], 'Unit':['m³','kg/m³','m³','kg','h','min','min']})
display(operations)


## 9. Route proposal and next steps
Continue with numbered notebooks 02–04, or run `python run_pipeline.py --planning-date 2026-08-17 --max-sites 24 --evaluate`. The cap is an explicit demonstration subset; deferred sites remain in the exports. Test a full fleet with a suitable Gurobi license, validate coordinates/travel assumptions, and compare matched service before claiming savings.

In [ ]:
print(f'Analyzed {len(df):,} recorded visits at {df.Location_id.nunique()} locations.')
print(f'Cutoff-specific profiles: {len(clustered_df)} bins; observations strictly before {PLANNING_DATE.date()}.')
print('Continue to 03_fill_prediction.ipynb and 04_gurobi_route_optimization.ipynb.')
